## Задание 1

Реализуйте класс с полностью инкапсулированным состоянием, используя name mangling и property, обеспечив валидацию при изменении атрибутов и демонстрируя, как Python скрывает "приватные" данные.

In [77]:
class Encapsulated:
    def __init__(self, value):
        self.__value = None
        self.value = value
    
    @property
    def value(self):
        return self.__value
    
    @value.setter
    def value(self, value):
        if not isinstance(value, int):
            raise TypeError("Value not integer")
        self.__value = value

obj = Encapsulated(10)
print(obj.value)
print(obj._Encapsulated__value)

10
10


## Задание 2

Создайте иерархию классов с абстрактным базовым классом (ABC) и абстрактными методами, демонстрируя использование модуля abc. Реализуйте интерфейсы и их проверку через isinstance и issubclass.

In [2]:
from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass
    
class Dog(Animal):
    def sound(self):
        return "Woof"
    
class Cat(Animal):
    def sound(self):
        return "Meow"


try:
    animal = Animal()
except:
    print("Нельзя создать объект класса Animal")

dog = Dog()
cat = Cat()

print(dog.sound(), cat.sound())
print(issubclass(Dog, Animal), issubclass(Cat, Animal))
print(isinstance(dog, Animal), isinstance(cat, Animal))

Нельзя создать объект класса Animal
Woof Meow
True True
True True


## Задание 3

Реализуйте дженерик-функцию (duck typing) с использованием протоколов (PEP 544) и типовых подсказок, чтобы показать полиморфизм без наследования.

In [ ]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> None:
        ...

class Circle:
    def draw(self) -> None:
        print('Draw Circle')

class Square:
    def draw(self) -> None:
        print('Draw Square')

def paint(obj: Drawable):
    obj.draw()

paint(Circle())
paint(Square())

Draw Circle
Draw Square


## Задание 4

Опишите и продемонстрируйте работу метода getattribute в отличие от getattr, реализуйте логирование всех обращений к атрибутам объекта, исследуйте возможные зацикливания.

In [4]:
class Logger:
    def __getattribute__(self, name):
      print(f'__getattribute__: request for attribute {name!r}')
      return object.__getattribute__(self, name)

    def __getattr__(self, name):
      print(f'__getattr__: {name!r} not found')
      return f'missing attribute: {name}'

obj = Logger()
obj.x = 5
print(obj.x)
print(obj.y)

__getattribute__: request for attribute 'x'
5
__getattribute__: request for attribute 'y'
__getattr__: 'y' not found
missing attribute: y


Здесь видно, что для атрибута obj.x изначально вызвался 
```__getattribute__```, где значение получается через 
```object.__getattribute__```. Для несуществующего атрибута obj.y сначала вызывается ```__getattribute__```, но получив ошибку в ```object.__getattribute__``` ввиду ненайденного атрибута, вызывается ```__getattr__```, где уже логируется запрос к несозданному атрибуту.  

## Задание 5

Создайте класс, который использует метакласс, объясните как метаклассы влияют на создание классов в Python, и реализуйте контролируемое изменение класса через метакласс.

In [14]:
class Meta(type):
    def __new__(mcs, name, bases, namespace):
        namespace['created_by_meta'] = True
        if 'describe' not in namespace:
            def describe(self):
                return f'{name} class was modified by Meta'
            namespace['describe'] = describe
        return super().__new__(mcs, name, bases, namespace)

class GoodClass(metaclass=Meta):
    def describe(self):
        return "This class was constructed from Meta, but extended also"
    
class SecondGoodClass(metaclass=Meta):
    pass

g = GoodClass()
print(g.created_by_meta) 
print(g.describe())
d = SecondGoodClass()
print(d.describe())

True
This class was constructed from Meta, but extended also
SecondGoodClass class was modified by Meta


Класс ```GoodClass``` получает метакласс ```Meta```, в котором определено создание атрибута ```created_by_meta```, и метода ```describe``` если его не существует в определении класса. Класс ```SecondGoodClass``` также имеет метакласс ```Meta```, но в этом случае метод ```describe``` уже не определён в описании класса, поэтому берётся стандартный метод из метакласса. Сами классы создаются по цепочке экземпляр - класс - метакласс. То есть экземпляр создаётся из класса, а класс создаётся из метакласса.

## Задание 6

Реализуйте класс с дескрипторами данных и неданных, объясните разницу между ними и механизм вызова методов get, set, delete, особенно при наследовании.

In [45]:
class DataDescriptor:
    def __set_name__(self, owner, name):
        self.storage_name = '_' + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.storage_name, '<empty>')
    
    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError('data_desc must be int')
        instance.__dict__[self.storage_name] = value

    def __delete__(self, instance):
        instance.__dict__.pop(self.storage_name, None)

class NonDataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return 'value from non-data descriptor'

class MyClass2:
    data_desc = DataDescriptor()
    non_data_desc = NonDataDescriptor()

obj = MyClass2()
obj.data_desc = 10
print(obj.data_desc)
print(obj.non_data_desc)

10
value from non-data descriptor


Data descriptor имеет ```__get__``` и ```__set__``` и ```__delete__```, поэтому он имеет приоритет над атрибутами экземпляра. Non-data descriptor содержит только ```__get__```, поэтому может быть перекрыт значением из ```instance.__dict__```. При чтении Python сначала проверяет data descriptor в классе, затем атрибуты экземпляра, затем non-data descriptor и обычные атрибуты класса. При наследовании дескрипторы участвуют в поиске атрибутов через MRO, поэтому логика сохраняется и в наследственных классах.

## Задание 7

Напишите класс с поддержкой множественного наследования, демонстрирующий работу C3-линеаризации MRO на сложном примере с 3+ уровнями наследования и пересечениями.

In [52]:
class A:
    def method(self):
        print('A.method')

class B(A):
    def method(self):
        print('B.method before super()')
        super().method()
        print('B.method after super()')

class C(A):
    def method(self):
        print('C.method before super()')
        super().method()
        print('C.method after super()')

class D(B, C):
    def method(self):
        print('D.method before super()')
        super().method()
        print('D.method after super()')
d = D()
d.method()
print(D.__mro__)

D.method before super()
B.method before super()
C.method before super()
A.method
C.method after super()
B.method after super()
D.method after super()
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


## Задание 8

Реализуйте класс с пользовательскими dunder-методами: new, init, call, del, repr, str, и объясните, как Python вызывает их в цикле жизни объекта.

In [62]:
class LifeCycle:
    def __new__(cls, name='base'):
        print('__new__: creating instance')
        return super().__new__(cls)
    
    def __init__(self, name='base'):
        print('__init__: initializing instance')
        self.name = name

    def __call__(self):
        print(f'__call__: {self.name} called like method')
      
    def __repr__(self):
        return f"LifeCycle({self.name!r})"
    
    def __str__(self):
        return f'LifeCycle object: {self.name}'

    def __del__(self):
        print(f'__del__: deleting {self.name}')

obj = LifeCycle(name='object')
print(obj)
print(repr(obj))
obj()
del obj

__new__: creating instance
__init__: initializing instance
LifeCycle object: object
LifeCycle('object')
__call__: object called like method
__del__: deleting object


При создании объекта сначала вызывается ```__new__``` для создания экземпляра объекта в памяти. Потом вызывается ```__init__```, который инициализирует состояние объекта. Дальше вызывается ```__call__``` при вызове ```print(obj)```. При вызове ```print(repr(obj))``` вызывается ```__repr__```. При вызове объекта как метода вызывается ```__call__```, а метод ```__del__``` вызывается при удалении объекта ```del obj```.

## Задание 9

Создайте контекстный менеджер с помощью специальных методов enter и exit, используйте его вместе с классом, в котором присутствуют методы с разграничением прав доступа.

https://habr.com/ru/articles/739326/

In [72]:
class Access:
    def __init__(self):
        self._secret = 'scary secret'
        self._access_granted = False
    
    def __enter__(self):
        self._access_granted = True
        return self._secret
    
    def __exit__(self, exc_type, exc, tb):
        self._access_granted = False
        return False
    
    def read_secret(self):
        if not self._access_granted:
            return 'Access denied!'
        return self._secret

    @property
    def secret(self):
        if self._access_granted:
            return self._secret
        return '<hidden>'

obj = Access()
print(obj.secret)
print(obj.read_secret())
with obj as secret:
    print(secret)
    print(obj.read_secret())
print(obj.secret)

<hidden>
Access denied!
scary secret
scary secret
<hidden>


## Задание 10

Создайте класс, объекты которого могут быть отслежены с помощью слабых ссылок (weakref). Реализуйте систему, которая хранит слабые ссылки на все созданные объекты класса и автоматически удаляет их из списка при уничтожении объектов. Продемонстрируйте это поведение, выводя текущее количество живых объектов.

In [73]:
import gc
import weakref


class TrackedObject:
    _refs = []

    def __init__(self, name):
        self.name = name
        self.__class__._refs.append(
            weakref.ref(self, self.__class__._remove_ref)
        )

    @classmethod
    def _remove_ref(cls, dead_ref):
        cls._refs = [ref for ref in cls._refs if ref is not dead_ref]

    @classmethod
    def alive_count(cls):
        cls._refs = [ref for ref in cls._refs if ref() is not None]
        return len(cls._refs)

    @classmethod
    def alive_objects(cls):
        cls._refs = [ref for ref in cls._refs if ref() is not None]
        return [ref().name for ref in cls._refs]


a = TrackedObject("A")
b = TrackedObject("B")

print(TrackedObject.alive_count(), TrackedObject.alive_objects())

del a
gc.collect()

print(TrackedObject.alive_count(), TrackedObject.alive_objects())


2 ['A', 'B']
1 ['B']
